In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import kurtosis, spearmanr
import itertools
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. PARAMETRIC CONFIGURATION
# ==============================================================================
# 1. Get the directory where the current Jupyter Notebook is located
# Path.cwd() gets the current working directory of the notebook
notebook_dir = Path.cwd() 

# 2. Navigate up to the parent directory (crypto-perpetual-arbitrage)
project_dir = notebook_dir.parent 

# 3. Navigate into the 'data' folder
DATA_DIR = project_dir / "data"

# 4. Point to the specific parquet file
MASTER_PARQUET = DATA_DIR / "BERA_TAGGED_MASTER.parquet"

# (Optional) Verify the paths are correct
print(f"Data Directory: {DATA_DIR}")
print(f"Master Parquet: {MASTER_PARQUET}")

GLOBAL_START = pd.Timestamp('2025-03-01 00:00:00')
GLOBAL_END = pd.Timestamp('2025-12-31 23:59:00')

ROLLING_WINDOW_MINS = 1440  # 24-Hour Ex-Ante Lookback
MIN_VALID_MINS = 720        # Require at least 12h of clean data inside the 24h window
ADJ_SPREAD_PREFIX = 'spread_padj_idx_mid_avg'

# ==============================================================================
# PART 1: MACRO STYLIZED FACTS
# ==============================================================================
def run_part1_stylized_facts(df, all_pairs):
    print("\n" + "="*90)
    print(" 📘 PART 1: MACRO STYLIZED FACTS & STATIONARITY")
    print(" ⚙️  APPROACH: STATIC EX-POST (FULL SAMPLE)")
    print(" 🎯 GOAL: Profile unconditional spatial properties across the entire 2025 horizon.")
    print("          Proves global cointegration using static, full-year ADF tests.")
    print("="*90)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        
        y_continuous = df[pair_col].where(active_mask)
        x_continuous = y_continuous.shift(1)
        acf1 = y_continuous.corr(x_continuous)
        
        clean_series = y_continuous.dropna()
        if len(clean_series) < 10000: continue
        
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"
        
        adf_series = clean_series.iloc[::10]
        adf_stat, p_value, _, _, _, _ = adfuller(adf_series, maxlag=5)
        is_stat = "Yes***" if p_value < 0.01 else "No"
        
        results.append({
            'Pair': pair_name, 'Mean(bps)': clean_series.mean(), 'StdDev(bps)': clean_series.std(),
            'Kurtosis': kurtosis(clean_series), 'ACF(1)': acf1, 'ADF P-Val': p_value,
            'Stationary': is_stat, 'Active Days': len(clean_series) / 1440.0
        })

    res_df = pd.DataFrame(results).sort_values('StdDev(bps)')
    fmt = {'Mean(bps)': "{:.3f}", 'StdDev(bps)': "{:.3f}", 'Kurtosis': "{:.2f}", 
           'ACF(1)': "{:.4f}", 'ADF P-Val': "{:.4f}", 'Active Days': "{:.1f}"}
    for col, f in fmt.items(): res_df[col] = res_df[col].apply(f.format)
    print(res_df.to_string(index=False))

# ==============================================================================
# PART 2: TACTICAL SIGNAL FREQUENCY 
# ==============================================================================
def run_part2_signal_frequency(df, all_pairs):
    print("\n" + "="*90)
    print(" 📘 PART 2: TACTICAL SIGNAL FREQUENCY")
    print(" ⚙️  APPROACH: DYNAMIC ROLLING EX-ANTE (1440m WINDOW)")
    print(" 🎯 GOAL: Compares basic Z-Scores against Ornstein-Uhlenbeck S-Scores.")
    print("          Proves the S-Score filters out multi-day macro drift via rolling AR(1).")
    print("="*90)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"

        y = df[pair_col].where(active_mask)
        x = y.shift(1)
        
        roll_mean = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()
        roll_std = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).std()
        roll_cov = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).cov(x)
        roll_var_x = x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        roll_var_y = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        
        rho_1 = (roll_cov / roll_var_x).clip(lower=1e-5, upper=0.999)
        m = (roll_mean - rho_1 * x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()) / (1 - rho_1)
        var_e = (roll_var_y - (rho_1**2) * roll_var_x).clip(lower=1e-8)
        sigma_eq = np.sqrt(var_e / (1 - rho_1**2))
        
        z_scores = ((y - roll_mean) / roll_std).dropna()
        s_scores = ((y - m) / sigma_eq).dropna()
        
        if len(s_scores) < 10000: continue
        
        z_3s, s_3s = (np.abs(z_scores) > 3.0).sum(), (np.abs(s_scores) > 3.0).sum()
        
        results.append({
            'Pair': pair_name, 'Naive Z > 3σ': z_3s, 'OU S-Score > 3σ': s_3s,
            'Naive Z > 4σ': (np.abs(z_scores) > 4.0).sum(), 'OU S-Score > 4σ': (np.abs(s_scores) > 4.0).sum(),
            'Filtered Toxicty %': (1 - (s_3s / max(z_3s, 1))) * 100
        })

    res_df = pd.DataFrame(results).sort_values('OU S-Score > 3σ', ascending=False)
    res_df['Filtered Toxicty %'] = res_df['Filtered Toxicty %'].apply(lambda x: f"{x:.1f}%")
    print(res_df.to_string(index=False))

# ==============================================================================
# PART 3: OU MODEL VALIDATION
# ==============================================================================
def run_part3_ou_validation(df, all_pairs):
    print("\n" + "="*90)
    print(" 📘 PART 3: OU STOCHASTIC MODEL VALIDATION & RESIDUAL DIAGNOSTICS")
    print(" ⚙️  APPROACH: STATIC EX-POST (FULL SAMPLE REGRESSION)")
    print(" 🎯 GOAL: OLS Regression of AR(1) limits. Mean Delta near 0 proves the theoretical")
    print("          equilibrium tightly tracks physical empirical order book centers.")
    print("="*90)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"

        y = df[pair_col].where(active_mask)
        x = y.shift(1)
        df_reg = pd.DataFrame({'y': y, 'x': x}).dropna()
        if len(df_reg) < 10000: continue

        model = sm.OLS(df_reg['y'], sm.add_constant(df_reg['x'])).fit()
        alpha, beta = model.params.iloc[0], model.params.iloc[1]
        
        if beta >= 1 or beta <= 0: continue
        theo_mean = alpha / (1 - beta)
        emp_mean = df_reg['y'].mean()
        lb_pval = acorr_ljungbox(model.resid, lags=[10], return_df=True)['lb_pvalue'].iloc[0]
        
        results.append({
            'Pair': pair_name, 'AR(1) Beta': beta, 'Theo Mean(μ)': theo_mean,
            'Emp Mean(x̄)': emp_mean, 'Mean Delta(Δ)': abs(theo_mean - emp_mean), 'Ljung-Box P-Val': lb_pval
        })

    res_df = pd.DataFrame(results).sort_values('AR(1) Beta')
    fmt = {'AR(1) Beta': "{:.4f}", 'Theo Mean(μ)': "{:.3f}", 'Emp Mean(x̄)': "{:.3f}", 
           'Mean Delta(Δ)': "{:.4f}", 'Ljung-Box P-Val': "{:.4f}"}
    for col, f in fmt.items(): res_df[col] = res_df[col].apply(f.format)
    print(res_df.to_string(index=False))

# ==============================================================================
# PART 4: MODEL USEFULNESS & RANK CORRELATION (THEORETICAL VS EMPIRICAL TTR)
# ==============================================================================
def calculate_anchored_empirical_ttr(y_series, m_series, s_scores, sigma_thresh):
    """ Vectorized forward-search to find true physical reversion time (capped at 48 hours) """
    triggers = np.where(np.abs(s_scores) >= sigma_thresh)[0]
    
    clean_triggers = []
    last_t = -60
    for t in triggers:
        if t - last_t >= 60:
            clean_triggers.append(t)
            last_t = t
            
    if not clean_triggers: return np.nan
    
    ttrs = []
    y_vals = y_series.values
    m_vals = m_series.values
    s_vals = s_scores.values
    max_len = len(y_vals)
    
    for t in clean_triggers:
        anchor_m = m_vals[t]
        direction = -1 if s_vals[t] > 0 else 1
        
        max_search = min(t + 2880, max_len) # 48 Hour cap
        forward_y = y_vals[t+1 : max_search]
        
        if direction == -1: # Price is high, needs to cross below anchor_m
            crossings = np.where(forward_y < anchor_m)[0]
        else: # Price is low, needs to cross above anchor_m
            crossings = np.where(forward_y > anchor_m)[0]
            
        if len(crossings) > 0:
            ttrs.append(crossings[0] + 1) # +1 translates index back to physical minutes
            
    # Median ignores 48-hour tail drifts, extracting the true 'typical' structural elasticity
    return np.median(ttrs) if ttrs else np.nan

def run_part4_theoretical_vs_empirical(df, all_pairs):
    print("\n" + "="*100)
    print(" 📘 PART 4: MODEL USEFULNESS & RANK CORRELATION (THEORETICAL VS EMPIRICAL TTR)")
    print(" ⚙️  APPROACH: HYBRID (Ex-Ante Rolling Theo vs Ex-Post Realized Execution)")
    print("="*100)
    print(" 📖 DEFINITIONS & ASSUMPTIONS:")
    print(" - Data Source: Strictly uses the Adjusted (Theoretical) Spread. This isolates the ")
    print("   structural cointegration from raw physical order book friction, ensuring an ")
    print("   'Apples-to-Apples' comparison with the AR(1) Theoretical Half-Life.")
    print(" - Start Trigger (t): The minute the 24h Rolling Ex-Ante S-Score hits >= 1.0, 2.0, or 3.0-Sigma.")
    print(" - Stop Target: The TTR timer stops when the spread crosses the 24h Ex-Ante moving average ")
    print("   (m), which is permanently anchored/frozen at its value at minute t.")
    print(" - The Two-Phase Decay Paradox: TTR is intentionally longer than the 1-Minute Drop.")
    print("   The 1m Drop captures the violent 'Microstructure Shock' (refilling empty order books), ")
    print("   while the TTR measures the slower 'Structural Bleed' of the actual capital imbalance.")
    print("="*100)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"
        
        # 🛡️ THE FIX: NaN-Mask first to maintain continuous time for exact TTR counting
        y = df[pair_col].where(active_mask)
        x = y.shift(1)
        
        # Rolling Theoretical AR(1) Parameters
        roll_mean = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()
        roll_cov = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).cov(x)
        roll_var_x = x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        roll_var_y = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        
        rho_1_series = (roll_cov / roll_var_x).clip(lower=1e-5, upper=0.999)
        avg_rho_1 = rho_1_series.mean()
        
        kappa = -np.log(avg_rho_1) if avg_rho_1 > 0 and pd.notna(avg_rho_1) else np.nan
        theo_hl = np.log(2) / kappa if pd.notna(kappa) and kappa > 0 else np.nan
        
        # Rolling S-Scores for trigger generation
        m = (roll_mean - rho_1_series * x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()) / (1 - rho_1_series)
        var_e = (roll_var_y - (rho_1_series**2) * roll_var_x).clip(lower=1e-8)
        sigma_eq = np.sqrt(var_e / (1 - rho_1_series**2))
        s_scores = ((y - m) / sigma_eq)
        
        if len(s_scores.dropna()) < 10000: continue
        
        # 2. Anchored Empirical TTRs (Bulk & Tail)
        ttr_1s = calculate_anchored_empirical_ttr(y, m, s_scores, 1.0)
        ttr_2s = calculate_anchored_empirical_ttr(y, m, s_scores, 2.0)
        ttr_3s = calculate_anchored_empirical_ttr(y, m, s_scores, 3.0)
        
        results.append({
            'Pair': pair_name,
            'Theo Half-Life (m)': theo_hl,
            'Anchored TTR 1σ': ttr_1s,
            'Anchored TTR 2σ': ttr_2s,
            'Anchored TTR 3σ': ttr_3s
        })

    if not results:
        print("[!] No pairs met the minimum sample size constraint for Part 4.")
        return

    res_df = pd.DataFrame(results).sort_values('Theo Half-Life (m)')
    
    # Calculate Spearman Rank Correlations (ignoring NaNs)
    corr_1s = spearmanr(res_df['Theo Half-Life (m)'], res_df['Anchored TTR 1σ'], nan_policy='omit')[0]
    corr_2s = spearmanr(res_df['Theo Half-Life (m)'], res_df['Anchored TTR 2σ'], nan_policy='omit')[0]
    corr_3s = spearmanr(res_df['Theo Half-Life (m)'], res_df['Anchored TTR 3σ'], nan_policy='omit')[0]
    
    fmt = lambda x: f"{x:.1f}" if pd.notna(x) else "N/A"
    for c in ['Theo Half-Life (m)', 'Anchored TTR 1σ', 'Anchored TTR 2σ', 'Anchored TTR 3σ']:
        res_df[c] = res_df[c].apply(fmt)
            
    print(res_df.to_string(index=False))
    print("="*100)
    print(f" 📈 SPEARMAN RANK CORRELATION (Theo HL vs 1σ Bulk Regime TTR): {corr_1s:.3f}")
    print(f" 📈 SPEARMAN RANK CORRELATION (Theo HL vs 2σ Void Regime TTR): {corr_2s:.3f}")
    print(f" 📈 SPEARMAN RANK CORRELATION (Theo HL vs 3σ Tail Regime TTR): {corr_3s:.3f}")
    print("="*100)


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    print("🚀 Ingesting Master Dataset...")
    df = pd.read_parquet(MASTER_PARQUET).loc[GLOBAL_START:GLOBAL_END]
    exchanges = [c.split('_')[-1] for c in df.columns if c.startswith('mid_')]
    all_pairs = list(itertools.combinations(exchanges, 2))
    
    run_part1_stylized_facts(df, all_pairs)
    run_part2_signal_frequency(df, all_pairs)
    run_part3_ou_validation(df, all_pairs)
    run_part4_theoretical_vs_empirical(df, all_pairs)

Data Directory: C:\Users\tz_ho\crypto-perpetual-arbitrage\data
Master Parquet: C:\Users\tz_ho\crypto-perpetual-arbitrage\data\BERA_TAGGED_MASTER.parquet
🚀 Ingesting Master Dataset...

 📘 PART 1: MACRO STYLIZED FACTS & STATIONARITY
 ⚙️  APPROACH: STATIC EX-POST (FULL SAMPLE)
 🎯 GOAL: Profile unconditional spatial properties across the entire 2025 horizon.
          Proves global cointegration using static, full-year ADF tests.
     Pair Mean(bps) StdDev(bps) Kurtosis ACF(1) ADF P-Val Stationary Active Days
BINA/BYBI    -1.794       5.973    12.27 0.9275    0.0000     Yes***       137.4
BYBI/GATE     3.149       6.866    24.43 0.9451    0.0000     Yes***       137.4
BINA/GATE     0.223       7.219   102.76 0.9337    0.0000     Yes***       305.0
BYBI/KUCO    -8.286       7.254    21.52 0.9338    0.0000     Yes***       137.2
BINA/OKEX    -2.952       7.390   170.73 0.9436    0.0000     Yes***       304.0
BYBI/OKEX    -2.360       7.559   117.91 0.9699    0.0000     Yes***       136.2
GAT